# Lesson 0: The Nudge, Linear Recurrences, and Why Every ML Algorithm Secretly Uses Them

**What this is:** the one idea underneath gradient descent, learning rates, running averages and Q-learning. You already wrote it. This lesson takes your own function apart, names every piece, derives the math by hand, and has you type each piece of code yourself.

**How to use it:** this notebook runs top to bottom. Whenever you see a code block, there is an empty code cell right below it. Type the code into that cell yourself (don't copy-paste), run it, then compare with the expected output shown after. The class you build in Step 5 keeps growing: later steps tell you to go back and add methods to that same cell.  
Every "Try this" is for a new cell of your own, so the original stays intact.

**Your function, for reference:**

```python
def nudge(estimate, target, alpha):
    return estimate + alpha * (target - estimate)
```

---

## The map

| Step | Question it answers |
|---|---|
| 1 | What does `nudge` actually do, in plain terms? |
| 2 | Why does your loop's output match `1 - .5**i`? (And why were the rows off by one?) |
| 3 | What is shrinking each step, and by how much? |
| 4 | How do I rewrite `nudge` as "multiply, then add"? (This is the *linear recurrence* form.) |
| 5 | Build the general tool: a `LinearRecurrence` class |
| 6 | Where does the sequence end up, and how do I compute that without looping? |
| 7 | A formula for step *n* directly, with no loop |
| 8 | What happens for *other* multipliers: slow, flip-flopping, exploding? |
| 9 | The same shape in coffee cooling and loans |
| 10 | The same shape in ML: gradient descent, running averages, Q-learning |
| 11 | Challenge: a different bowl |

## Step 1: What `nudge` does

`nudge` takes three numbers:

- `estimate`: where you are now
- `target`: where you want to be
- `alpha`: what **fraction of the remaining gap** to close this step

`(target - estimate)` is the gap. `alpha * gap` is the piece of the gap you close. Adding it to `estimate` moves you that far toward the target.

Type this:

```python
def nudge(estimate, target, alpha):
    return estimate + alpha * (target - estimate)

print(nudge(0, 1, 0.5))
```

In [ ]:
# type it here

You should see `0.5`: you were at 0, the gap was 1, you closed half of it.

Now call it again from the new position:

```python
print(nudge(0.5, 1, 0.5))
```

In [ ]:
# type it here

`0.75`. The gap was 0.5, half of it is 0.25, so 0.5 + 0.25.

**Try this (on a copy):** keep `estimate=0, target=1` and swap `alpha` for `0.1`, then `1`, then `0`. Before you run each, predict the answer. What does `alpha=1` mean in words? What does `alpha=0` mean?

*(Answers: 0.1, 1, 0. With `alpha=1` you close the whole gap in one go. With `alpha=0` you never move.)*

## Step 2: Your loop, and the mystery line

Your original loop printed `start` and then `1 - .5**i`. Here's a version with clearer names. Type it:

```python
current_estimate = 0
nudge_count = 0

while abs(1 - current_estimate) > 0.0001:
    current_estimate = nudge(current_estimate, 1, 0.5)
    nudge_count += 1
    print(nudge_count, current_estimate, 1 - 0.5**nudge_count)
```

In [ ]:
# type it here

What each line does:

- `abs(1 - current_estimate) > 0.0001` is "keep going while we're still more than 0.0001 away from 1". `abs` is there because the distance can't be negative.
- Inside the loop we nudge, count the nudge, then print three things: how many nudges so far, where we are, and the value of `1 - 0.5**nudge_count`.

The first rows should be:

```
1 0.5 0.5
2 0.75 0.75
3 0.875 0.875
4 0.9375 0.9375
5 0.96875 0.96875
```

The last row is nudge 14: `0.99993896484375` in both columns. It took **14 nudges** to get within 0.0001 of 1.

**The two columns match.** So after *n* nudges the estimate is exactly `1 - 0.5**n`. That's your "turns out it's the same as this basic math function" discovery, and the rest of this lesson is *why*.

**A note on your original code:** it printed `start` *before* nudging but `1 - .5**i` was already for the *next* value. So `start` was always one row behind the formula (row 1 showed `0` next to `0.5`). The formula wasn't wrong, the printing was offset by one. Printing after nudging, as above, lines them up.

**Try this:** change `0.5` to `0.2` in the nudge call, and change the formula column to `1 - 0.8**nudge_count`. Do they still match? Why `0.8`? (Hold that thought. Step 4 explains it.)

## Step 3: What's actually shrinking

Don't look at the estimate. Look at the **error**: how far from the target you still are.

```python
current_estimate = 0
previous_error = 1 - current_estimate

for nudge_count in range(1, 6):
    current_estimate = nudge(current_estimate, 1, 0.5)
    current_error = 1 - current_estimate
    print(nudge_count, current_estimate, current_error, current_error / previous_error)
    previous_error = current_error
```

In [ ]:
# type it here

Output:

```
1 0.5 0.5 0.5
2 0.75 0.25 0.5
3 0.875 0.125 0.5
4 0.9375 0.0625 0.5
5 0.96875 0.03125 0.5
```

The last column is the same every time: **the error is multiplied by 0.5 each step**. That's the whole secret. "Close half the gap" and "multiply the error by 0.5" are the same statement. So after *n* steps the error is `0.5**n` (starting error 1), and the estimate is `1 - 0.5**n`. That's your formula.

For general `alpha` you close `alpha` of the gap, so what's *left* is `(1 - alpha)` of it. The error is multiplied by `1 - alpha` each step. That's why Step 2's exploration used `0.8` for `alpha = 0.2`.

## Step 4: Algebra: rewriting `nudge` as "multiply, then add"

Start from your function's body and expand it:

```
estimate + alpha * (target - estimate)
= estimate + alpha*target - alpha*estimate        (distribute alpha)
= (1 - alpha)*estimate + alpha*target             (group the estimate terms)
```

So `nudge` is secretly:

```
next_value = multiplier * current_value + offset
    where  multiplier = 1 - alpha
           offset     = alpha * target
```

For your numbers (`alpha = 0.5`, `target = 1`): `multiplier = 0.5`, `offset = 0.5`.

**That form is what "linear recurrence" means.** *Recurrence*: a rule that gets the next value from the previous one (so you can generate a whole sequence). *Linear*: the previous value is only multiplied by a constant and added to a constant. No squares, no square roots, no `if`s. The general first-order version is:

```
next = multiplier * current + offset
```

Type a quick check that the two forms agree:

```python
estimate = 0.3
alpha = 0.5
target = 1

print(nudge(estimate, target, alpha))
print((1 - alpha) * estimate + alpha * target)
```

In [ ]:
# type it here

Both lines should print `0.65`.

## Step 5: Build the general tool

Now a class. It holds the two numbers that define the recurrence, and can step once or run many steps. Type this into a new cell. This is the file we'll keep growing:

```python
class LinearRecurrence:
    def __init__(self, multiplier, offset):
        self.multiplier = multiplier
        self.offset = offset

    def step(self, current_value):
        return self.multiplier * current_value + self.offset

    def run(self, starting_value, number_of_steps):
        values = [starting_value]
        for _ in range(number_of_steps):
            values.append(self.step(values[-1]))
        return values
```

In [ ]:
# type it here

How it stores things: `__init__` runs when you create one, and saves the two numbers on the object as `self.multiplier` and `self.offset`. `step` is "one application of the rule". `run` keeps a list called `values`, starting with your start, and repeatedly appends `step(values[-1])` (`values[-1]` is the last item, i.e. the most recent value).

Use it to reproduce your original sequence:

```python
halving_recurrence = LinearRecurrence(multiplier=0.5, offset=0.5)
print(halving_recurrence.run(0, 6))
```

In [ ]:
# type it here

```
[0, 0.5, 0.75, 0.875, 0.9375, 0.96875, 0.984375]
```

Same numbers as your loop, produced by a general tool instead of one specific function.

**Try this:** `print(halving_recurrence.__dict__)`. You'll see `{'multiplier': 0.5, 'offset': 0.5}`, which is literally how the object stores its data.

## Step 6: Where does it end up? (the fixed point)

The sequence settles toward 1. A **fixed point** is a value that doesn't change when you apply the rule. Call it `x`. Then:

```
x = multiplier * x + offset
x - multiplier*x = offset
x * (1 - multiplier) = offset
x = offset / (1 - multiplier)
```

For your recurrence: `0.5 / (1 - 0.5) = 1`. ✓ The target.

For `nudge` generally: `(alpha*target) / (1 - (1 - alpha)) = (alpha*target)/alpha = target`. So the fixed point of a nudge is *always* its target, which is exactly what you'd hope.

Add this method **inside the class**, below `run`:

```python
    def fixed_point(self):
        return self.offset / (1 - self.multiplier)
```

**Add this *inside* the `LinearRecurrence` class cell from Step 5 (below `run`), then re-run that cell and re-run the `halving_recurrence = ...` cell.**

```python
print(halving_recurrence.fixed_point())   # 1.0
```

In [ ]:
# type it here

Note the division: if `multiplier` is exactly 1, you'd divide by zero. Remember that for Step 8.

## Step 7: A formula for step *n* with no loop

From Step 3 the error is multiplied by `multiplier` each step. Here's the derivation using the fixed point (call it `fixed`):

```
next_value = multiplier * current_value + offset      (the rule)
fixed      = multiplier * fixed + offset              (fixed point's rule)
```

Subtract the second line from the first:

```
next_value - fixed = multiplier * (current_value - fixed)
```

Read that: `(value - fixed)` is the error, and the new error is `multiplier` times the old error. Apply it *n* times:

```
value_after_n_steps = fixed + multiplier**n * (starting_value - fixed)
```

Add inside the class:

```python
    def value_after(self, starting_value, number_of_steps):
        fixed = self.fixed_point()
        return fixed + self.multiplier**number_of_steps * (starting_value - fixed)
```

**Add this *inside* the `LinearRecurrence` class cell from Step 5 (below `run`), then re-run that cell and re-run the `halving_recurrence = ...` cell.**

Check it against the loop:

```python
print([halving_recurrence.value_after(0, n) for n in range(7)])
```

In [ ]:
# type it here

```
[0.0, 0.5, 0.75, 0.875, 0.9375, 0.96875, 0.984375]
```

Identical to Step 5's list, and for your case `fixed = 1`, `multiplier = 0.5`, `starting_value = 0`, so the formula is `1 + 0.5**n * (0 - 1) = 1 - 0.5**n`. Your original formula, derived instead of guessed.

**Bonus question:** how many nudges until the error is below 0.0001? Solve `0.5**n < 0.0001`: `n > log(0.0001) / log(0.5) ≈ 13.29`, so 14. ✓ (matches the loop.) That's why a formula is worth having: it predicts the answer without running the loop.

## Step 8: Other multipliers, and what to expect

So far `multiplier = 0.5`. The multiplier controls everything. Type this and run it:

```python
test_cases = [
    (0.5, 0.5, 0),     # your original
    (-0.5, 3, 0),
    (1.1, 1, 0),
    (1, 2, 0),
    (-1.2, 1, 0),
    (0.9, 0.5, 0),
]

for multiplier, offset, starting_value in test_cases:
    recurrence = LinearRecurrence(multiplier, offset)
    print(multiplier, offset, [round(value, 3) for value in recurrence.run(starting_value, 6)])
```

In [ ]:
# type it here

What you should see (rounded):

| multiplier | offset | first values from 0 | what it does |
|---|---|---|---|
| 0.5 | 0.5 | 0, 0.5, 0.75, 0.875, … | smooth approach to 1 |
| -0.5 | 3 | 0, 3, 1.5, 2.25, 1.875, 2.063, 1.969 | **flip-flops** around 2 while shrinking |
| 1.1 | 1 | 0, 1, 2.1, 3.31, 4.641, 6.105, 7.716 | **grows without bound** |
| 1 | 2 | 0, 2, 4, 6, 8, 10, 12 | steady drift, no fixed point |
| -1.2 | 1 | 0, 1, -0.2, 1.24, -0.488, 1.586, -0.903 | **flip-flops and grows** |
| 0.9 | 0.5 | 0, 0.5, 0.95, 1.355, 1.72, 2.048, 2.343 | smooth but *slow* toward 5 |

The rule they all follow, straight from "the error is multiplied by `multiplier` each step":

- **|multiplier| < 1**: error shrinks, so you converge to the fixed point. (Closer to 0 = faster.)
- **multiplier > 0**: error keeps its sign, so you approach from one side, smoothly.
- **multiplier < 0**: error flips sign each step, so you jump back and forth across the fixed point.
- **|multiplier| > 1**: error grows, so you run away.
- **multiplier = 1**: error never shrinks. With a nonzero offset you drift forever and there's no fixed point (try `LinearRecurrence(1, 2).fixed_point()`: division by zero).

Notice something about the `1.1` row: its fixed point is `1 / (1 - 1.1) = -10`. Start *exactly* at -10 and you stay there forever. Start anywhere else and you run away from it. A fixed point that pushes things away is called **unstable**. A fixed point with |multiplier| < 1 is **stable**.

**Try this (on a copy):** rewrite the first row as a nudge, `LinearRecurrence(multiplier=1 - alpha, offset=alpha * target)`, using `alpha=0.9` and `target=1`. Predict what the sequence will look like before you run it. (Hint: `multiplier = 0.1`.) Then try `alpha=1.5`. What's the multiplier now, and what does the table say should happen?

## Step 9: Same shape, other places

Two real situations that are *exactly* `next = multiplier * current + offset`. Since your class is general, they're one-liners.

**A cooling coffee.** Each minute the coffee loses 10% of the gap between its temperature and the room's (20°C):

```python
coffee = LinearRecurrence(multiplier=0.9, offset=2.0)
print(coffee.fixed_point())                         # 20.0, the room temperature
print([round(t, 2) for t in coffee.run(90, 10)])
```

In [ ]:
# type it here

```
[90, 83.0, 76.7, 71.03, 65.93, 61.33, 57.2, 53.48, 50.13, 47.12, 44.41]
```

Why `offset = 2.0`? This *is* `nudge(temperature, 20, 0.1)`: multiplier `1 - 0.1 = 0.9`, offset `0.1 * 20 = 2`. The fixed point is the room temperature, as promised.

**A loan.** 5% interest per period (multiplier 1.05) and you pay 100 each period (offset -100):

```python
loan = LinearRecurrence(multiplier=1.05, offset=-100)
print(round(loan.fixed_point(), 2))                                 # 2000.0
print([round(balance, 2) for balance in loan.run(1500, 6)])
print([round(balance, 2) for balance in loan.run(2500, 6)])
print([round(balance, 2) for balance in loan.run(2000, 6)])
```

In [ ]:
# type it here

```
[1500, 1475.0, 1448.75, 1421.19, 1392.25, 1361.86, 1329.95]
[2500, 2525.0, 2551.25, 2578.81, 2607.75, 2638.14, 2670.05]
[2000, 2000.0, 2000.0, 2000.0, 2000.0, 2000.0, 2000.0]
```

Here `multiplier = 1.05 > 1`, so the fixed point (2000) is **unstable**: pay 100 on a 2000 balance and you only just cover the interest (5% of 2000 = 100). Start below and you pay it off; start above and you spiral. Same table from Step 8, now with meaning.

## Step 10: Where this lives in machine learning

Now the payoff.

### Gradient descent is `nudge` in disguise

Picture the simplest "bowl" error: `loss(w) = 0.5 * (w - target)**2`. Its slope is `(w - target)`. Gradient descent says: step *downhill*, a distance of `learning_rate * slope`:

```
w_new = w - learning_rate * (w - target)
      = w + learning_rate * (target - w)        (flip both signs)
      = nudge(w, target, learning_rate)
```

**The learning rate is alpha.** Everything you learned about `alpha` applies to learning rates.

Type this to confirm the slope claim numerically (the "wiggle" idea from the intuition series):

```python
def bowl_loss(weight, target):
    return 0.5 * (weight - target) ** 2

def bowl_slope(weight, target):
    return weight - target

weight = 3.0
target = 1.0
tiny_wiggle = 0.0001

wiggled_slope = (bowl_loss(weight + tiny_wiggle, target) - bowl_loss(weight, target)) / tiny_wiggle
print(wiggled_slope)                    # about 2.00005
print(bowl_slope(weight, target))       # 2.0
```

In [ ]:
# type it here

The wiggle estimate and the formula agree, which is what a derivative *is*: "how much does the loss change per tiny wiggle of the weight".

Now watch learning rate act as the multiplier (`multiplier = 1 - learning_rate`):

```python
for learning_rate in (0.5, 1.9, 2.0, 2.1):
    weight = 0.0
    history = []
    for _ in range(6):
        weight = weight - learning_rate * bowl_slope(weight, 1.0)
        history.append(round(weight, 4))
    print(learning_rate, history)
```

In [ ]:
# type it here

```
0.5 [0.5, 0.75, 0.875, 0.9375, 0.9688, 0.9844]
1.9 [1.9, 0.19, 1.729, 0.3439, 1.5905, 0.4686]
2.0 [2.0, 0.0, 2.0, 0.0, 2.0, 0.0]
2.1 [2.1, -0.21, 2.331, -0.4641, 2.6105, -0.7716]
```

Match each row to Step 8: `lr=0.5` has multiplier `0.5` (smooth). `lr=1.9` has multiplier `-0.9` (flip-flops, shrinking slowly). `lr=2.0` has multiplier `-1` (flip-flops forever, neither shrinking nor growing). `lr=2.1` has multiplier `-1.1` (flip-flops *and grows*, it diverges). **That is the whole story of "learning rate too high"**, explained by one number.

### Running average

Say you want the average of numbers as they arrive, without storing them all. After seeing the *n*-th number `x`:

```python
running_mean = 0
for count, new_number in enumerate([4, 8, 6], start=1):
    running_mean = running_mean + (1 / count) * (new_number - running_mean)
    print(count, running_mean)
```

In [ ]:
# type it here

```
1 4.0
2 6.0
3 6.0
```

(Check: the real mean of 4, 8, 6 is 6. ✓) That's `nudge(running_mean, new_number, 1/count)`, a nudge where alpha *shrinks* as `1/n`. Honest caveat: with a changing alpha it's no longer a *constant* multiplier, so the closed form in Step 7 doesn't directly apply. But the one-step shape is the same.

If you hold alpha *constant* (say 0.5), you get an **exponential moving average**: recent numbers matter most. Unroll it and the weight on the number from `k` steps ago is `alpha * (1 - alpha)**k`:

```python
alpha = 0.5
print([alpha * (1 - alpha) ** steps_ago for steps_ago in range(5)])
```

In [ ]:
# type it here

```
[0.5, 0.25, 0.125, 0.0625, 0.03125]
```

Each older number counts half as much as the one after it, which is "the error is multiplied by 0.5 each step" wearing a different hat.

### Q-learning (your maze)

The update rule is:

```
new_value = old_value + alpha * (reward + discount * best_next_value - old_value)
```

Look at the shape: `old + alpha * (SOMETHING - old)`. That's `nudge(old_value, reward + discount * best_next_value, alpha)`. The "target" is just a number computed from the reward and the next state's value, and each table cell is being nudged toward it.

### The pattern

| ML thing | `estimate` | `target` | `alpha` |
|---|---|---|---|
| Gradient descent on a bowl | the weight | the bowl's bottom | learning rate |
| Running mean | current mean | newest number | 1/n |
| Moving average | current average | newest number | fixed fraction |
| Q-learning | table cell | reward + discounted next value | learning rate |

Real losses aren't perfect bowls, and real networks have many weights. But *near* a minimum a smooth loss looks like a bowl in each direction, and the flip-flop-and-explode behavior you saw in Step 8 is the same one that wrecks real training runs.

## Step 11: Challenge: a steeper bowl

Try this *without* looking at the solution section below.

Suppose the loss is `loss(w) = 3 * (w - 2)**2`. Its slope is `6 * (w - 2)`. Gradient descent is `w = w - learning_rate * 6 * (w - 2)`, starting at `w = 0`.

1. Write that update as a call to `nudge(...)`. What are `estimate`, `target`, and `alpha`?
2. What is the multiplier, in terms of `learning_rate`?
3. Which learning rate lands exactly on 2 in a single step?
4. Above which learning rate does it start to diverge?
5. Predict the first few values for `learning_rate = 0.2`, then run it to check.

Write your answers on paper first, then type the loop and compare.

In [ ]:
# challenge: your attempt here

## Where to go from here

Once Steps 1-9 feel solid, the interesting exploration is: **what changes when there are two weights instead of one?** Two numbers can have *different* multipliers, and the biggest one decides whether training survives. Go and probe that yourself: make two copies of the bowl with different steepness, try one learning rate on both, and see what you can predict from this lesson. Then search for "learning rate and curvature" or "condition number" to see what people have named the thing you find.

# Solutions to Step 11 (no peeking until you've tried it)

1. `w = nudge(w, 2, 6 * learning_rate)`. Expand it to check: `w + 6*lr*(2 - w) = w - 6*lr*(w - 2)`. ✓ So `estimate = w`, `target = 2`, `alpha = 6 * learning_rate`.
2. `multiplier = 1 - alpha = 1 - 6 * learning_rate`.
3. One step lands exactly when `multiplier = 0`: `learning_rate = 1/6 ≈ 0.1667`. (Every step closes the whole gap.)
4. Divergence starts when `|multiplier| > 1`. Since `multiplier` falls below -1 when `1 - 6*lr < -1`, i.e. `lr > 1/3 ≈ 0.333`.
5. With `lr = 0.2`: `multiplier = 1 - 1.2 = -0.2`, so it flip-flops around 2 and shrinks fast.

```python
for learning_rate in (1/6, 0.2, 0.3, 0.34):
    weight = 0.0
    history = []
    for _ in range(6):
        weight = weight - learning_rate * 6 * (weight - 2)
        history.append(round(weight, 4))
    print(learning_rate, history)
```

```
0.1667 [2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
0.2    [2.4, 1.92, 2.016, 1.9968, 2.0006, 1.9999]
0.3    [3.6, 0.72, 3.024, 1.1808, 2.6554, 1.4757]
0.34   [4.08, -0.1632, 4.2497, -0.3397, 4.4333, -0.5306]
```

`lr = 0.3` has multiplier `-0.8`: still converging, but slowly and jumpily. `lr = 0.34` has multiplier `-1.04`: just past the edge, so it slowly blows up. A steeper bowl made the safe learning rate *smaller* (1/3 here, 2 for the earlier bowl). That's why "the right learning rate" depends on the problem.